In [15]:
from langchain_ollama.llms import OllamaLLM
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

In [20]:
def generate_response(question):
    
    inf_model=OllamaLLM(model="Eomer/gpt-3.5-turbo")
    
    myTemplate=ChatPromptTemplate.from_messages(
        
        [
            ("system","You are helpful {assistant} who can answer to the user questions"),
            ("user","Tell me about {question}")
            
        ]
    )
    
    my_chain=myTemplate|inf_model |StrOutputParser()
    
    response=my_chain.invoke({"assistant":"financial adviser","question":question})
    
    return response
    
    

In [21]:
result=generate_response("retirement savings")
result

"Of course, I'd be happy to help! Retirement savings is an important aspect of planning for your financial future. It's never too early or too late to start saving for retirement, and there are several options available to help you reach your goals. Here are some key things to consider:\n\n1. Start Early: The earlier you start saving for retirement, the better. Compound interest can work in your favor over a longer period of time, allowing your savings to grow exponentially. Even small contributions to a retirement account can add up significantly over time.\n2. Take Advantage of Tax-Advantaged Accounts: Retirement accounts such as 401(k), IRA, and Roth IRA offer tax benefits that can help your savings grow faster. Contributions to these accounts are tax-deductible, and the money grows tax-deferred until you withdraw it in retirement.\n3. Set a Retirement Goal: Determine how much money you'll need in retirement based on your desired lifestyle and expenses. This will help guide your sav

In [22]:
from langchain_community.document_loaders import PyPDFLoader

loader=PyPDFLoader("/Users/sreesekhar/IAI/AgenticGenerativeAI/week9/lab/data/chase_banking.pdf")
documents=loader.load()

print(documents)

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2024-10-03T09:29:06-04:00', 'author': 'JPMorgan Chase Bank', 'keywords': 'Chase; total; checking; guide to your account; ada; (PDF)', 'moddate': '2024-10-07T09:59:35-04:00', 'subject': 'Chase Total Checking - A Guide To Your Account', 'title': 'Chase Total Checking - A Guide To Your Account (PDF)', 'trapped': '/Unknown', 'source': '/Users/sreesekhar/IAI/AgenticGenerativeAI/week9/lab/data/chase_banking.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='HAVE QUESTIONS? CALL US AT 1-800-935-9935 (WE ACCEPT OPERATOR RELAY CALLS)  • WANT MORE INFO?  SEE THE DEPOSIT ACCOUNT AGREEMENT\n1\nCHASE TOTAL CHECKING\n®\nA GUIDE TO YOUR ACCOUNT †\nIt’s important that you understand how your Chase Total Checking account works. \nWe’ve created this Guide to explain the fees and some key terms of your personal account.\nMONTHLY \nSERVICE FEE*\nMonthly Service Fee $12

In [23]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import  HuggingFaceEmbeddings

embedding_llm=HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/Users/sreesekhar/IAI/AgenticGenerativeAI/week9/lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
faiss_db=FAISS.from_documents(documents,embedding_llm)

In [25]:
retriever=faiss_db.as_retriever()

In [26]:
qa_template="""You are an Financial Expert for question-answer tasks.
               Use the following peice of retrieved context to answer the questions.
               If you don't know the answer, Just say I don't know.
               Use only 5 sentences maximumb to keep the answer concice.
               
               question: {question}
               context: {context}
               Answer : 

"""

In [29]:
qa_prompt=ChatPromptTemplate.from_template(qa_template)

In [28]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough

In [41]:
def format_myDocs(docs):
    
    return "\n\n--------".join(doc.page_content for doc in docs)

In [38]:
 inf_llm=OllamaLLM(model="Eomer/gpt-3.5-turbo")

In [44]:
rag_chain=(
    RunnableParallel(context=retriever|format_myDocs, question=RunnablePassthrough()) | qa_prompt | inf_llm)

In [45]:
rag_chain.invoke("How to avoid over draft fees for checking account")

'This document outlines the posting order for transactions deposited into a Chase bank account. The posting order is the order in which transactions are applied to the account, and it helps customers better manage their accounts.\n\nDuring nightly processing, Chase posts transactions in the following order:\n\n1. First, any previous day adjustments are made, and deposits are added to the account.\n2. Second, Chase subtracts transactions in chronological order based on the date and time of authorization or presentation. If multiple transactions have the same date and time, they are posted in high to low dollar order.\n3. Third, there are some transactions that Chase cannot process automatically or until nightly processing is complete. These include Overdraft Protection transfers or transfers to maintain target balances in other accounts.\n4. Finally, fees are assessed last.\n\nThe document also notes that when deposits are available (see Funds Availability Policy in the Deposit Account 